# Postgres DB Inspector

This notebook connects to the project's Postgres database and shows quick sanity checks (tables, row counts, recent rows).

If you're running this on your Mac (not inside Docker), make sure Postgres is exposed on `localhost:5432` via `docker compose`.


In [ ]:
# If you don't have these installed locally, run this once:
# !pip -q install pandas psycopg2-binary

import os
from pathlib import Path

import pandas as pd
import psycopg2


def load_env_file(path: str = ".env") -> dict:
    p = Path(path)
    if not p.exists():
        return {}
    out = {}
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        out[k.strip()] = v.strip().strip(""'")
    return out


env = {**load_env_file(".env"), **os.environ}

host = env.get("DB_HOST", "localhost")
# In docker-compose the API/airflow use DB_HOST=postgres, but from your laptop that should be localhost.
if host == "postgres":
    host = "localhost"

conn = psycopg2.connect(
    host=host,
    port=int(env.get("DB_PORT", "5432")),
    user=env.get("DB_USER"),
    password=env.get("DB_PASSWORD"),
    dbname=env.get("DB_NAME"),
)
conn.autocommit = True
print("connected to", host)



In [ ]:
def q(sql: str, params=None):
    return pd.read_sql_query(sql, conn, params=params)

q(
    "SELECT table_schema, table_name
"
    "FROM information_schema.tables
"
    "WHERE table_schema IN ('raw', 'processed', 'features')
"
    "ORDER BY table_schema, table_name"
)



In [ ]:
q('SELECT COUNT(*) AS n FROM raw.eth_ohlcv')



In [ ]:
q('SELECT COUNT(*) AS n FROM processed.eth_ohlcv_1h')



In [ ]:
q('SELECT COUNT(*) AS n FROM features.eth_features')



In [ ]:
q(
    "SELECT *
"
    "FROM processed.eth_ohlcv_1h
"
    "ORDER BY ts DESC
"
    "LIMIT 10"
)



In [ ]:
q(
    "SELECT *
"
    "FROM features.eth_features
"
    "ORDER BY ts DESC
"
    "LIMIT 10"
)



In [ ]:
conn.close(); print('closed')

